# Compact Multi-Layer Perceptron (MLP) Baseline for HIGGS Classification

**Author:** Rachit Gupta (neural baseline lead) | **Project:** Higgs Event Lab | **Module:** MID-04

This notebook implements and demonstrates the compact neural network baseline for classifying simulated particle collision events in the HIGGS benchmark. It investigates whether regularized deep neural architectures can effectively learn representations from low-level particle kinematic measurements compared to high-level physicist-engineered invariant masses.

> **Scientific boundaries:** All evaluations in this study classify simulated events generated by Monte Carlo tools. Model outputs represent classification scores on benchmark events, not physical discovery claims.

## Download the complete original HIGGS data

This cell downloads the complete approximately 11-million-row UCI HIGGS dataset when it is missing. The MLP benchmark below trains on the complete raw file, not on a processed subset.


In [ ]:
from pathlib import Path
import sys
import urllib.request
import zipfile
import shutil

root = Path.cwd()
while not (root / "src" / "higgs_event_lab").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

raw_path = root / "data" / "raw" / "HIGGS.csv.gz"
processed_path = raw_path
full_row_count = 11_000_000
output_dir = root / "outputs"
raw_path.parent.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

url = "https://archive.ics.uci.edu/static/public/280/higgs.zip"

if not raw_path.exists():
    print("Downloading the complete HIGGS dataset from UCI...")
    zip_path = raw_path.parent / "higgs_download.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        gz_name = [name for name in archive.namelist() if name.endswith(".csv.gz")][0]
        with archive.open(gz_name) as source, open(raw_path, "wb") as target:
            shutil.copyfileobj(source, target)
    zip_path.unlink()

print("Database source:", url)
print("Complete raw data ready:", raw_path)
print("Expected complete rows:", f"{full_row_count:,}")


## 1. Environment and Package Initialization

**Question:** Which libraries and modules are required to run the deep learning benchmark?

**Why it matters:** To guarantee strict experimental reproducibility and modular design, all model layers, training loops, and data loading contracts reside in tested package modules rather than loose notebook script blocks.

**Explanation:** We import PyTorch for building the neural network, alongside NumPy and Pandas for tensor operations. We also import our project modules `features`, `protocol`, and `neural` to enforce schema constraints and split isolation.

**Expected output:** Confirmation of the PyTorch version and available device acceleration.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from higgs_event_lab.features import FEATURE_GROUPS, get_feature_columns
from higgs_event_lab.protocol import SplitConfig, create_stratified_splits
from higgs_event_lab.neural import MLPConfig, build_mlp, run_feature_group_benchmarks

print("PyTorch version:", torch.__version__)
print("CUDA accelerator available:", torch.cuda.is_available())

## 2. Multi-Layer Perceptron Architecture and Regularization

**Question:** How is the neural network structured to prevent overfitting and ensure fast, stable convergence?

**Why it matters:** Unregularized deep networks easily overfit high-dimensional collision data. Batch normalization stabilizes internal covariate shift during gradient updates, while dropout prevents co-adaptation of hidden features.

**Explanation:** The compact MLP architecture applies sequential hidden blocks containing a linear projection, one-dimensional batch normalization, ReLU activation, and twenty percent dropout. The final layer outputs an unconstrained binary logit.

**Expected output:** Layer-by-layer architectural printout and parameter counts.

In [ ]:
config = MLPConfig(hidden_sizes=(64, 32), dropout=0.20, learning_rate=1e-3)
sample_model = build_mlp(input_features=28, config=config)
total_params = sum(param.numel() for param in sample_model.parameters())
print(sample_model)
print("Total network parameters:", total_params)

## 3. The 21 Low-Level and 7 High-Level Feature Groups

**Question:** How are the twenty-eight input variables partitioned across kinematic and derived feature groups?

**Why it matters:** The central scientific question of the benchmark is evaluating whether neural networks require human-engineered invariant masses or can learn equivalent or superior decision boundaries directly from raw detector quantities.

**Explanation:** Low-level features comprise momenta, pseudorapidity, azimuthal angles, and b-tagging metrics. High-level features comprise invariant masses calculated from reconstructed decay subsystems.

**Expected output:** Verified counts and column names for the low-level, high-level, and complete feature sets.

In [ ]:
for group_name in ("low_level", "high_level", "all"):
    columns = get_feature_columns(group_name)
    print(f"{group_name:<11}: {len(columns)} features -> {list(columns[:3])}...")

## 4. Reproducible stratified splitting without data leakage

**Question:** How do we partition all approximately 11 million events into training, validation and held-out test subsets fairly?

**Why it matters:** The complete-data benchmark uses the shared 70% / 15% / 15% stratified protocol with seed 42. Scaling the row count changes the measured result, so the source and exact count are recorded in every artifact.

**Expected output:** Exact full-data row counts for each disjoint partition.


In [ ]:
from higgs_event_lab.data import load_higgs_sample

batch = load_higgs_sample(processed_path, n_rows=full_row_count, feature_group="all")
split = create_stratified_splits(batch.y.to_numpy(), SplitConfig(seed=42))
X = batch.X
y = batch.y.to_numpy()

print(f"Dataset: {len(X):,} rows loaded from the complete raw file")
print(f"Features: {X.shape[1]} columns")
print(f"Signal fraction: {y.mean():.4f}")
print(f"Split sizes — Train: {len(split.train):,}  Val: {len(split.validation):,}  Test: {len(split.test):,}")
del batch, X, y, split


## 5. Training Across All Three Feature Sets

**Question:** How does model performance vary across low-level, high-level, and combined feature representations?

**Why it matters:** Comparing model metrics across the three feature groups reveals the relative importance of engineered domain features versus raw sensor measurements.

**Explanation:** We train identical MLP architectures across each subset using AdamW with early stopping monitored on validation ROC-AUC. When training halts, the best historical model weights are automatically restored.

**Expected output:** Completed training epoch count and test evaluation metrics for each feature group.

In [ ]:
config = MLPConfig(
    hidden_sizes=(64, 32),
    dropout=0.20,
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    early_stopping_patience=5,
    batch_size=1024,
    seed=42,
    device="auto",
)
benchmark_results = run_feature_group_benchmarks(
    data_path=processed_path,
    row_count=full_row_count,
    output_base_dir=output_dir,
    config=config,
)
for group_name, result in benchmark_results.items():
    metrics = result["test"]["metrics"]
    print(f"{group_name:<11} epochs={len(result['history']):2d} device={result['device']:<4} "
          f"ROC-AUC={metrics['roc_auc']:.4f} PR-AUC={metrics['pr_auc']:.4f}")


## 6 — Artifact Export for Dashboard and Reports

The benchmark helper writes each measured MLP run to `outputs/mlp_*` with a `metrics.json` file containing genuine training history and a `test_predictions.csv` file containing held-out predictions.


## 6.1 Export learning curves

The training module stores genuine per-epoch loss and validation ROC-AUC in each MLP `metrics.json`. This cell plots those saved histories for direct inclusion in the report.


In [ ]:
import json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for group_name in ("low_level", "high_level", "all"):
    metadata = json.loads((output_dir / f"mlp_{group_name}" / "metrics.json").read_text())
    history = metadata["training"]["history"]
    epochs = [item["epoch"] for item in history]
    axes[0].plot(epochs, [item["train_loss"] for item in history], label=group_name)
    axes[1].plot(epochs, [item["validation_roc_auc"] for item in history], label=group_name)
axes[0].set(xlabel="Epoch", ylabel="Training loss", title="MLP training loss")
axes[1].set(xlabel="Epoch", ylabel="Validation ROC-AUC", title="MLP validation learning")
for axis in axes:
    axis.legend()
fig.tight_layout()
fig.savefig(output_dir / "mlp_learning_curves.png", dpi=160)
plt.show()
print("Learning curves saved to:", output_dir / "mlp_learning_curves.png")


## 7 — Verify exported full-data artifacts

The complete approximately 11-million-row MLP runs are exported to `outputs/mlp_*` as `higgs-experiment-v1` artifacts. This final cell confirms that each run contains metadata and held-out predictions.


In [ ]:
expected_run_ids = ["mlp_low_level", "mlp_high_level", "mlp_all"]

print("Artifact verification:")
all_ok = True
for run_id in expected_run_ids:
    run_dir = output_dir / run_id
    metadata_ok = (run_dir / "metrics.json").is_file()
    predictions_ok = (run_dir / "test_predictions.csv").is_file()
    all_ok = all_ok and metadata_ok and predictions_ok
    print(f"  {run_id:<18} metrics.json={metadata_ok} predictions={predictions_ok}")

if not all_ok:
    raise FileNotFoundError("One or more MLP artifacts are missing; rerun the training cell.")
print("All three measured MLP artifacts are ready for the report and dashboard.")
